In [ ]:
!pip install -r requirements.txt

In [2]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoConfig, TrainingArguments, Trainer, BertModel, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from pypinyin import pinyin, Style
import torch
import torch.nn as nn
import numpy as np
import evaluate
import pandas as pd
import os
import pronouncing

/opt/miniconda3/envs/Chinese_pun/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/Chinese_pun/lib/python3.12/site-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


In [3]:
# ============================================================
# DEVICE CONFIGURATION - Set USE_GPU to True if GPU available
# ============================================================
import torch

USE_GPU = False  # Set to True if you have GPU (CUDA/MPS)

# Detect available device
if USE_GPU:
    if torch.cuda.is_available():
        device = "cuda"
        print("✅ GPU (CUDA) detected and enabled")
    elif torch.backends.mps.is_available():
        device = "mps"  # Mac with Apple Silicon
        print("✅ GPU (MPS) detected and enabled")
    else:
        device = "cpu"
        print("⚠️ GPU requested but not available, falling back to CPU")
else:
    device = "cpu"
    print("✅ Using CPU (set USE_GPU=True to enable GPU)")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device.upper()}")

✅ Using CPU (set USE_GPU=True to enable GPU)
PyTorch version: 2.10.0
Device: CPU


## PIYING PREPROCESSING

In [4]:
map_dict = {
    'ㄅ': 'p', 'ㄆ': 'ph', 'ㄇ': 'm', 'ㄈ': 'f', 'ㄉ': 't','ㄊ': 'th', 'ㄋ': 'n', 
    'ㄌ': 'l', 'ㄍ': 'k', 'ㄎ': 'kh', 'ㄏ': 'h', 'ㄐ': 'ts', 'ㄑ': 'tsh', 'ㄒ': 's', 
    'ㄓ': 'tsr', 'ㄔ': 'tshr', 'ㄕ': 'sr', 'ㄖ': 'jr', 'ㄗ': 'ts', 'ㄘ': 'tsh', 'ㄙ': 's',
    'ㄚ': 'a', 'ㄛ': 'o', 'ㄜ': 'o', 'ㄝ': 'e', 'ㄞ': 'ai', 'ㄟ': 'ei', 'ㄠ': 'au', 'ㄡ': 'ou', 
    'ㄢ': 'an', 'ㄣ': 'en', 'ㄤ': 'ang', 'ㄥ': 'eng', 'ㄦ': 'er', 'ㄧ': 'i', 'ㄨ': 'u', 'ㄩ': 'yu'
}

alpabet_to_pinyin = {
    "B": "ㄅ", "P": "ㄆ", "M": "ㄇ", "F": "ㄈ", "D": "ㄉ", "T": "ㄊ", "N": "ㄋ", "L": "ㄌ",
    "G": "ㄍ", "K": "ㄎ", "HH": "ㄏ", "JH": "ㄐ", "CH": "ㄔ", "ZH": "ㄓ", "SH": "ㄕ", 
    "S": "ㄙ", "Z": "ㄗ", "TH": "ㄊ", "DH": "ㄉ", "V": "ㄈ", "W": "ㄨ", "Y": "ㄧ", 
    "R": "ㄖ", "NG": "ㄥ", "AA": "ㄚ", "AE": "ㄝ", "AH": "ㄜ", "AO": "ㄛ", "AW": "ㄠ",
    "AY": "ㄞ", "EH": "ㄟ", "EY": "ㄟ", "IH": "ㄧ", "IY": "ㄧ", "OW": "ㄡ", "OY": "ㄡㄧ",
    "ER": "ㄦ", "UH": "ㄨ", "UW": "ㄨ"
}

In [5]:
# Create phonetic vocabulary
unique_phonetics = set(map_dict.values())
p_vocab = {phonetic: i + 2 for i, phonetic in enumerate(sorted(unique_phonetics))}
p_vocab["[PAD]"] = 0
p_vocab["[UNK]"] = 1

In [6]:
def handle_chinese(text):
    result = pinyin(text, style=Style.BOPOMOFO)
    piying = ""
    for char_list in result:
        for char in char_list[0]:
            if map_dict.get(char):
                piying += map_dict[char]
    return piying

def handle_english(text):
    phones = pronouncing.phones_for_word(text)
    p_rep = []
    if phones:
        for symbol in phones[0].split():
            base_symbol = symbol.rstrip("012")
            if base_symbol in alpabet_to_pinyin:
                p_rep.append(map_dict[alpabet_to_pinyin[base_symbol]])
    return "".join(p_rep)

def piying_preprocessing(text):
    if text.isascii():
        return handle_english(text.lower())
    return handle_chinese(text)


## Context Model

### Define Model

In [7]:
import torch.nn as nn
from transformers import BertModel, BertPreTrainedModel, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback, PreTrainedConfig


In [8]:
class PinyinBertClassifier(nn.Module):
    def __init__(self, config, pinyin_vocab_size):
        super().__init__()
        self.config = config 
        self.bert = BertModel.from_pretrained("bert-base-chinese", config=config)
        self.pinyin_embeddings = nn.Embedding(pinyin_vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, input_ids, attention_mask=None, pinyin_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state 
        p_embeds = self.pinyin_embeddings(pinyin_ids)
        
        # Fusion of semantic and phonetic features
        fused_output = sequence_output + p_embeds
        pooled_output = fused_output[:, 0, :] # [CLS] token
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

### Data Preprocessing

In [9]:
data_path = "./data/combined_pun_nonpun.csv"
df = pd.read_csv(data_path).dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(pd.DataFrame({"text": X_train.values, "label": y_train.values}), preserve_index=False),
    "validation": Dataset.from_pandas(pd.DataFrame({"text": X_val.values, "label": y_val.values}), preserve_index=False)
})

tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

In [10]:
def preprocess_function(examples):
    result = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    all_pinyin_ids = []
    for text in examples["text"]:
        p_string = piying_preprocessing(text)
        ids = [p_vocab.get(char, 1) for char in p_string]
        ids = [0] + ids[:126] + [0] # Alignment with [CLS] and [SEP]
        ids += [0] * (128 - len(ids))
        all_pinyin_ids.append(ids)
    result["pinyin_ids"] = all_pinyin_ids
    return result

tokenized_dataset = dataset.map(preprocess_function, batched=True)


Map: 100%|██████████| 148/148 [00:00<00:00, 5925.46 examples/s]


### Train

In [11]:
config = AutoConfig.from_pretrained("bert-base-chinese", num_labels=2)
model = PinyinBertClassifier(config, pinyin_vocab_size=len(p_vocab)).to(device)

accuracy = evaluate.load("accuracy")
f1_macro = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_macro.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1637.61it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
args = TrainingArguments(
    output_dir="./models/toxic_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16 if device != "cpu" else 2,
    num_train_epochs=5 if device != "cpu" else 2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

/opt/miniconda3/envs/Chinese_pun/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.095502,0.979730,0.979519
2,0.045118,0.076726,0.972973,0.972729


/opt/miniconda3/envs/Chinese_pun/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=592, training_loss=0.04159134948575819, metrics={'train_runtime': 194.0986, 'train_samples_per_second': 6.1, 'train_steps_per_second': 3.05, 'total_flos': 0.0, 'train_loss': 0.04159134948575819, 'epoch': 2.0})

### Inference

In [13]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
    
    # Process Pinyin for a single sentence
    p_string = piying_preprocessing(text)
    p_ids = torch.tensor([[pinyin_vocab.get(p, 1) for p in p_string.split()]])
    # ... (Add padding/CLS/SEP logic same as training) ...

    with torch.no_grad():
        out = model(input_ids=inputs["input_ids"], pinyin_ids=p_ids)
        return torch.argmax(out["logits"], dim=-1)

In [14]:
predict("你的笑话真有趣")

NameError: name 'pinyin_vocab' is not defined

In [ ]:
# 11. Confusion Matrix & Classification Report
predictions = trainer.predict(dataset["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["Non-Pun", "Pun"]))

# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Pun", "Pun"],
            yticklabels=["Non-Pun", "Pun"],
            cbar_kws={'label': 'Count'})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Pun Classification")
plt.tight_layout()
plt.show()

print("\n✅ Model training completed!")
print(f"Model saved to: {output_dir}")